<a href="https://colab.research.google.com/github/faizan-tnvr004/AI-Based-Meeting-Transcript-Organizer/blob/main/AI_Deliverable_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

we have divided our work into three chunks

part 1: creating the dataset, building the text cleaning function , encoding labels, TF-IDF vectorization, and the train/test split.

part 2:builds the neural network layer by layer with detailed explanations of ReLU, Dropout, Softmax

part 3:  train the model, plots accuracy/loss curves, builds the TextRank summarizer using NetworkX, evaluates with Precision/Recall/F1, draws the confusion matrix, runs a full end-to-end demo,

In [2]:
#faizan
#using NLTK to do tokenization
import nltk
nltk.download('punkt')        #sentence/word splitter
nltk.download('punkt_tab')    #updated version of punkt
nltk.download('stopwords')    #list of filler words to ignore
nltk.download('wordnet')      #dictionary for finding word roots


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [11]:
#faizan
import pandas as pd
import numpy as np
import re
import json

from datasets import load_dataset

#using this for text processing
from nltk.corpus import stopwords #this is for filler words
from nltk.stem import WordNetLemmatizer #this is to reduce words
from nltk.tokenize import word_tokenize #this is for splitting text

#sklear tools

from sklearn.preprocessing import LabelEncoder #converts text labels to number
from sklearn.feature_extraction.text import TfidfVectorizer #converts text to numerical matrices
from sklearn.model_selection import train_test_split #to split data
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score # for measuring
from sklearn.metrics.pairwise import cosine_similarity #measure sentence similarity

#neural network
import tensorflow as tf
from tensorflow.keras.models import Sequential #stacking layers
from tensorflow.keras.layers import Dense, Dropout, Activation
from tensorflow.keras.callbacks import EarlyStopping


#graphs
import networkx as nx #for graphs
from sklearn.metrics.pairwise import cosine_similarity #measure sentence similarity

#visualization
import seaborn as sns #statistical plots
import matplotlib.pyplot as plt #core plots

tf.random.set_seed(42)
np.random.seed(42)

print("all libraries imported")
print(f"tensorFlow: {tf.__version__}")


all libraries imported
tensorFlow: 2.19.0


In [16]:
#faizan
#importing QMSum dataset

import urllib.request
import json

# Define the direct links to the raw GitHub files
repo_urls = {
    'train': 'https://raw.githubusercontent.com/Yale-LILY/QMSum/main/data/ALL/jsonl/train.jsonl',
    'validation': 'https://raw.githubusercontent.com/Yale-LILY/QMSum/main/data/ALL/jsonl/val.jsonl',
    'test': 'https://raw.githubusercontent.com/Yale-LILY/QMSum/main/data/ALL/jsonl/test.jsonl'
}

qmsum = {}

# Open network connections and parse the JSONL files into lists
for split_name, file_url in repo_urls.items():
    with urllib.request.urlopen(file_url) as network_response:
        qmsum[split_name] = [json.loads(line) for line in network_response]

print(f"\nQMSum loaded successfully from GitHub!")
print(f"Splits available: {list(qmsum.keys())}")
print(f"Train records:      {len(qmsum['train'])}")
print(f"Validation records: {len(qmsum['validation'])}")
print(f"Test records:       {len(qmsum['test'])}")

# Inspect the structure of ONE record to understand the format
sample = qmsum['train'][0]
print(f"\nKeys in each record: {list(sample.keys())}")
print(f"Number of transcript turns in first record: {len(sample['meeting_transcripts'])}")
print(f"Number of queries in first record: {len(sample['specific_query_list'])}")

# Show one example query
print("\nExample query from first record:")
query_sample = sample['specific_query_list'][0]  # first query

print(f"  Query: {query_sample['query']}")
print(f"  Relevant spans: {query_sample['relevant_text_span']}")


QMSum loaded successfully from GitHub!
Splits available: ['train', 'validation', 'test']
Train records:      162
Validation records: 35
Test records:       35

Keys in each record: ['topic_list', 'general_query_list', 'specific_query_list', 'meeting_transcripts']
Number of transcript turns in first record: 506
Number of queries in first record: 6

Example query from first record:
  Query: How Did Project Manager and User Interface introduce the prototype of the remote control?
  Relevant spans: [['0', '39']]


In [21]:
# Keywords that signal a DECISION query
DECISION_KEYWORDS = [
    'decide', 'decided', 'decision', 'agree', 'agreed', 'agreement',
    'approve', 'approved', 'resolve', 'resolved', 'conclude', 'concluded',
    'settle', 'settled', 'vote', 'voted', 'confirm', 'confirmed',
    'choose', 'chose', 'chosen', 'select', 'selected', 'finalize', 'finalized'
]

# Keywords that signal an ACTION ITEM query
ACTION_KEYWORDS = [
    'action', 'task', 'assign', 'assigned', 'responsible', 'follow up',
    'follow-up', 'todo', 'to-do', 'will do', 'should do', 'needs to',
    'take care', 'handle', 'complete', 'implement', 'prepare', 'schedule',
    'send', 'submit', 'update', 'review', 'write', 'create', 'build'
]

import re

def contains_keyword(text, keywords):
    return any(re.search(rf'\b{kw}\b', text) for kw in keywords)


def detect_label_from_query(query_text):
    """
    Given a query string, returns 'Decision', 'Action Item', or None.
    None means we shouldn't override the default Discussion label.
    """
    query_lower = query_text.lower()
    # Check for Decision keywords first (more specific)
    if contains_keyword(query_lower, DECISION_KEYWORDS):
        return 'Decision'
    # Then check for Action Item keywords
    if contains_keyword(query_lower, ACTION_KEYWORDS):
        return 'Action Item'
    return None  # Not a labeled category → remains Discussion


def extract_labeled_sentences_from_qmsum(dataset_split):
    """
    Processes all records in a QMSum split and returns a DataFrame
    with columns: 'sentence', 'label'
    """
    rows = []

    for record in dataset_split:
        turns = record['meeting_transcripts']  # List of {speaker, content} dicts
        queries = record['specific_query_list']  # List of {query, answer, relevant_text_span}

        # Step 1: Initialize every turn as Discussion
        # turn_labels[i] = label for turn index i
        turn_labels = ['Discussion'] * len(turns)

        # Step 2: Scan each query and label relevant turns
        for q in queries:
            query_text = q.get('query', '')
            spans = q.get('relevant_text_span', [])

            detected_label = detect_label_from_query(query_text)
            if detected_label is None:
                continue  # This query doesn't indicate a labeled category

            # Each span is [start_index, end_index] as STRINGS
            for span in spans:
                try:
                    start = int(span[0])
                    end = int(span[1])
                    # Label all turns in this range
                    for i in range(start, min(end + 1, len(turns))):
                        # Decision takes priority over Action Item if both match
                        if detected_label == 'Decision':
                            turn_labels[i] = 'Decision'
                        elif turn_labels[i] == 'Discussion':
                            turn_labels[i] = 'Action Item'
                except (ValueError, IndexError):
                    continue  # Skip malformed spans

        # Step 3: Collect labeled turns into rows
        for i, turn in enumerate(turns):
            content = turn.get('content', '').strip()
            if len(content.split()) > 2:  # Skip very short utterances ("Yeah", "Okay", etc.)
                rows.append({
                    'sentence': content,
                    'label': turn_labels[i]
                })

    return pd.DataFrame(rows)


# --- Run on all splits and combine ---
print("Extracting labeled sentences from all QMSum splits...")
df_train = extract_labeled_sentences_from_qmsum(qmsum['train'])
df_val   = extract_labeled_sentences_from_qmsum(qmsum['validation'])
df_test  = extract_labeled_sentences_from_qmsum(qmsum['test'])

# Combine all splits into one DataFrame
# (we will do our own train/test split later at 80/20)
df = pd.concat([df_train, df_val, df_test], ignore_index=True)

print(f"\n Extraction complete!")
print(f"Total labeled sentences: {len(df):,}")
print(f"\nLabel distribution:")
print(df['label'].value_counts())
print(f"\nClass balance (%)")
print((df['label'].value_counts() / len(df) * 100).round(1))
print("\nFirst 5 rows:")
df.head()

Extracting labeled sentences from all QMSum splits...

 Extraction complete!
Total labeled sentences: 94,320

Label distribution:
label
Discussion     90384
Decision        3357
Action Item      579
Name: count, dtype: int64

Class balance (%)
label
Discussion     95.8
Decision        3.6
Action Item     0.6
Name: count, dtype: float64

First 5 rows:


,sentence,label
0,Yep . Soon as I get this . Okay . This is our ...,Discussion
1,The prototype discussion .,Discussion
2,The prototype yeah . Do you need a {disfmarker...,Discussion
3,No . {vocalsound},Discussion
4,{vocalsound} Can try to plug that in there,Discussion
5,There is our remo {gap} the banana .,Discussion
6,Um {vocalsound} yeah basically we we st went w...,Discussion
7,"Um but it would be held in such a fashion ,",Discussion
8,"where it is , obviously it wouldn't be that fl...",Discussion
9,Very nice .,Discussion


In [23]:
# Find the size of the smallest class
min_class_size = df['label'].value_counts().min()
if min_class_size < 500:
    cap = min_class_size
else:
    cap = 2000
print(f"Smallest class size before balancing: {min_class_size}")

# Cap at a reasonable number to keep training fast on Colab
# If the smallest class has fewer than 500, use that; otherwise cap at 2000
cap = min(min_class_size, 2000)
print(f"Capping each class at: {cap} samples")

# Sample each class
df_balanced = (
    df.groupby('label', group_keys=False)
      .apply(lambda x: x.sample(min(len(x), cap), random_state=42))
      .reset_index(drop=True)
)

# Shuffle the whole DataFrame
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nBalanced dataset size: {len(df_balanced):,} total sentences")
print("\nLabel distribution after balancing:")
print(df_balanced['label'].value_counts())
df_balanced.head()

Smallest class size before balancing: 579
Capping each class at: 579 samples

Balanced dataset size: 1,737 total sentences

Label distribution after balancing:
label
Action Item    579
Discussion     579
Decision       579
Name: count, dtype: int64


/tmp/ipykernel_2824/710818542.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), cap), random_state=42))


,sentence,label
0,We'll now go to Mr. Spengemann.,Action Item
1,Hmm ! That 's a little bit of a problem .,Discussion
2,Uh on the {disfmarker} you can have it on the ...,Decision
3,Well so we we can think about a well a a vocal...,Decision
4,I would say seven . It's hard to miss it .,Decision
